# L4 单层 KV cache-path · LRU 与冻结 rank 策略
在 Colab 的「修改运行时类型」中选 **L4 GPU**，然后「全部运行」。出现上传框时选择已有的 **KV_Cache_Agent_Handoff.zip**。最后一格会下载结果 ZIP。无需挂载 Drive、安装模型或配置账号密钥。请保持会话连接，等待 40/40 篇完成。

主实验使用原短上下文 held-out 的 40 篇文档、全部 28 层，按单层流从空缓存开始。每步请求相同的 K=64 页，B=128，每页 16 个 token，BF16 K/V 共 64 KiB。LRU 对照的是开发集冻结的完整策略（历史 rank 权重 δ=4，频率权重 β=2）。每次 miss 通过 pinned host → GPU staging → GPU slot 装入确定性合成 KV。计时从已准备好的 Top-K 开始，单位为 µs/layer-step。

In [ ]:
import csv, gzip, hashlib, json, os, shutil, subprocess, sys, tempfile
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
import torch, triton
assert torch.cuda.is_available(), "需要 GPU 运行时"
gpu_name = torch.cuda.get_device_name(0)
assert "L4" in gpu_name.split(), f"当前为 {gpu_name}，请切换到 NVIDIA L4 后重新全部运行"
ROOT = Path(tempfile.mkdtemp(prefix="kv-l4-", dir="/content"))
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
RESULT = ROOT/"results"
RESULT.mkdir()
print(gpu_name, torch.__version__, triton.__version__, torch.version.cuda)
print("输出目录", RESULT)
(RESULT/"nvidia-smi.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))

### 读取原始输入
只取交接包内两份数据 ZIP，并验证 SHA-256。原 smoke 归档包含计数和计时；访问顺序取自同一份短 trace 的四个任务 × 长度分组各自的首篇、第 0 层。

In [ ]:
spec = {
    "heldout-20261120T043814Z.zip": "24e5cfc913352ac8b85451dc77f0e4f72052b76584f449ad82b1ec8cc05c82f3",
    "gpu-layer-smoke-20261122T044930617682Z.zip": "b6021c1e61f0c2a6f8cb880d8ff58a945c73d40947dc6caae87017244d9837c2",
}
uploaded = files.upload()
assert len(uploaded)==1, "请选择完整的 KV_Cache_Agent_Handoff.zip"
from io import BytesIO
with ZipFile(BytesIO(next(iter(uploaded.values())))) as bundle:
    for filename, sha in spec.items():
        raw=bundle.read("KV_Cache_Agent_Handoff/data/short/"+filename)
        assert hashlib.sha256(raw).hexdigest()==sha, filename
        (ROOT/filename).write_bytes(raw)
del uploaded, raw
trace, archive = (ROOT/name for name in spec)

### 实验代码
下面三格写入配套 CPU 模拟器、Triton 单层实现、smoke 核对程序。模拟器按步记录 miss；GPU 每次回放均与其逐步比较。页内容检查单独运行，比较全部选中页、槽位映射及页内所有 BF16 值。

In [ ]:
%%writefile replay.py
"""Replay batched sparse-KV page selections. Core replay uses the standard library."""
import argparse
import csv
import json
from collections import Counter, defaultdict, deque
from pathlib import Path


def read_trace(path):
    """Load JSONL, preserving time order independently for each request and layer."""
    streams = defaultdict(list)
    with Path(path).open(encoding="utf-8") as source:
        for line_number, line in enumerate(source, 1):
            try:
                row = json.loads(line)
                request = row["request_id"]
                layer = row["layer_id"]
                step = row["decode_step"]
                pages = row["ranked_page_ids"]
                if not isinstance(request, str) or not request.strip():
                    raise ValueError("request_id must be a nonempty string")
                if type(layer) is not int or layer < 0:
                    raise ValueError("layer_id must be a nonnegative integer")
                stream = streams[request, layer]
                if type(step) is not int or step != len(stream):
                    raise ValueError(f"expected decode_step {len(stream)} for {(request, layer)}")
                if not isinstance(pages, list) or any(type(p) is not int or p < 0 for p in pages):
                    raise ValueError("ranked_page_ids must be a list of nonnegative integers")
                if len(pages) != len(set(pages)):
                    raise ValueError("duplicate page in the same step")
                stream.append(tuple(pages))
            except (KeyError, ValueError, TypeError) as error:
                raise ValueError(f"{path}, line {line_number}: {error}") from error
    if not streams:
        raise ValueError("empty trace")
    return dict(streams)


def replay_stream(steps, budget, policy, *, delta=0, beta=0, on_step=None):
    """Start one empty cache. A step is a set of simultaneous, compulsory admissions."""
    if policy not in ("lru", "belady", "rank_only", "weighted") or type(budget) is not int or budget < 1:
        raise ValueError("unsupported policy or invalid budget")
    future = defaultdict(deque)
    if policy == "belady":
        for step, pages in enumerate(steps):
            for page in pages:
                future[page].append(step)

    # Larger (step, was_hit, -page_id) means more recent in our LRU convention.
    resident = {}
    seen = set()
    frequency = Counter()
    ranks = {}
    rank_bonus = {}
    accesses = misses = 0
    for step, pages in enumerate(steps):
        selected = set(pages)
        if len(selected) != len(pages) or len(selected) > budget:
            raise ValueError(f"duplicate selection or insufficient budget at step {step}")
        hits = selected.intersection(resident)
        missing = selected - hits
        if policy == "belady":
            for page in selected:
                future[page].popleft()
        evict_count = max(0, len(resident) + len(missing) - budget)
        victims = ()
        if evict_count:
            candidates = resident.keys() - selected
            if policy == "rank_only":
                victims = sorted(candidates, key=lambda p: (-ranks[p], resident[p]))
            elif policy == "weighted":
                def priority(p):
                    score = resident[p][0] + delta * rank_bonus[p] + beta * min(frequency[p], 4)
                    return score, resident[p]
                victims = sorted(candidates, key=priority)
            elif policy == "lru":
                victims = sorted(candidates, key=resident.__getitem__)
            else:
                def farthest_first(page):
                    next_step = future[page][0] if future[page] else len(steps)
                    return -next_step, page
                victims = sorted(candidates, key=farthest_first)
            for page in victims[:evict_count]:
                del resident[page]
        if policy in ("rank_only", "weighted"):
            for rank, page in enumerate(pages, 1):
                ranks[page] = rank
                rank_bonus[page] = rank <= (len(pages) + 3) // 4
        if policy == "weighted" and beta:
            frequency.update(pages)
            if step >= 8:
                frequency.subtract(steps[step - 8])
        for page in selected:
            resident[page] = (step, page in hits, -page)
        if on_step is not None:
            on_step(step, pages, missing, victims[:evict_count])
        accesses += len(selected)
        misses += len(missing)
        seen.update(selected)

    return Counter(accesses=accesses, misses=misses, compulsory_misses=len(seen),
                   repeat_misses=misses - len(seen), repeat_accesses=accesses - len(seen))


def compare(streams, budgets):
    rows = []
    for budget in budgets:
        for policy in ("lru", "belady"):
            counts = Counter()
            for steps in streams.values():
                counts.update(replay_stream(steps, budget, policy))
            rows.append(dict(
                budget=budget, policy=policy, **counts,
                miss_rate=counts["misses"] / counts["accesses"] if counts["accesses"] else None,
                repeat_miss_rate=(counts["repeat_misses"] / counts["repeat_accesses"]
                                  if counts["repeat_accesses"] else None),
            ))
    return rows


def save_comparison(rows, out_dir, title):
    import matplotlib.pyplot as plt
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with (out_dir / "comparison.csv").open("w", newline="", encoding="utf-8") as output:
        writer = csv.DictWriter(output, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), layout="constrained")
    for axis, metric, label in zip(axes, ("miss_rate", "repeat_miss_rate"),
                                  ("All selected pages", "Previously selected pages")):
        for policy, name, style in (("lru", "LRU", "o-"), ("belady", "Bélády", "x--")):
            subset = sorted((r for r in rows if r["policy"] == policy), key=lambda r: r["budget"])
            rates = [float("nan") if r[metric] is None else 100 * r[metric] for r in subset]
            axis.plot([r["budget"] for r in subset], rates, style, label=name)
        axis.set(title=label, xlabel="Cache budget (pages per layer)", ylabel="Miss rate (%)")
        axis.set_xticks(sorted({r["budget"] for r in rows}))
        axis.set_ylim(bottom=0)
        axis.grid(axis="y", alpha=.25)
        axis.legend()
    fig.suptitle(title)
    fig.savefig(out_dir / "comparison.png", dpi=160)
    plt.close(fig)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("trace", type=Path)
    parser.add_argument("--budgets", type=int, nargs="+", default=[64, 128, 256])
    parser.add_argument("--out", type=Path, default=Path("results/replay"))
    args = parser.parse_args()
    rows = compare(read_trace(args.trace), args.budgets)
    save_comparison(rows, args.out, args.trace.stem)
    for row in rows:
        print(f"B={row['budget']:>3}  {row['policy']:<6}  misses {row['misses']}/{row['accesses']}"
              f"  first {row['compulsory_misses']}  repeat {row['repeat_misses']}/{row['repeat_accesses']}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile gpu_layer_benchmark.py
"""Measure the HiSparse-style layer cache path on frozen held-out traces."""
import argparse
import csv
import gzip
import hashlib
import json
import random
import statistics
import tempfile
from collections import Counter, defaultdict
from pathlib import Path
from time import perf_counter_ns
from zipfile import ZipFile

from replay import read_trace, replay_stream

BUDGET = 128
TOP_K = 64
PAGE_SHAPE = (16, 2, 8, 128)  # tokens, K/V, KV heads, head dimension
PAGE_BYTES = 2 * 16 * 2 * 8 * 128  # BF16
POLICIES = {"LRU": (0, 0), "rank_delta4_beta2": (4, 2)}


def define_resolver():
    import triton
    import triton.language as tl

    # Triton resolves tl as a module global, including when JIT compiles in Colab.
    globals()["tl"] = tl

    @triton.jit
    def resolve(Query, SlotPage, PageSlot, Last, WasHit, TopRank, History, Descriptor,
                Dest, step, PAGE_STRIDE: tl.constexpr, B: tl.constexpr, K: tl.constexpr,
                DELTA: tl.constexpr, BETA: tl.constexpr):
        lane = tl.arange(0, B)
        query_lane = lane < K
        wanted = tl.load(Query + lane, query_lane, other=0)
        hit_slot = tl.load(PageSlot + wanted, query_lane, other=-1)
        hit = query_lane & (hit_slot >= 0)
        miss = query_lane & ~hit
        # Count non-hits in bin zero, then subtract them (portable histogram API).
        bins = tl.histogram(tl.where(hit, hit_slot, 0), B)
        protected = bins > tl.where(lane == 0, tl.sum((~hit).to(tl.int32), axis=0), 0)

        slot_page = tl.load(SlotPage + lane)
        occupied = slot_page >= 0
        candidate = occupied & ~protected
        safe_page = tl.maximum(slot_page, 0)
        last = tl.load(Last + safe_page, candidate, other=0)
        was_hit = tl.load(WasHit + safe_page, candidate, other=0)
        score = last
        if DELTA:
            high = tl.load(TopRank + safe_page, candidate, other=0)
            score += DELTA * high
        if BETA:
            history = tl.load(History + safe_page, candidate, other=0)
            # Bit zero marks the last selection. Shift by age-1 to put t-1 at bit zero.
            age = tl.minimum(tl.maximum(step - last - 1, 0), 8)
            bits = (history << age) & 255
            freq = tl.full((B,), 0, tl.int32)
            for bit in tl.static_range(8):
                freq += (bits >> bit) & 1
            score += BETA * tl.minimum(freq, 4)

        # Lexicographic eviction order: score, last step, hit flag, descending page ID.
        # 512 covers last_step*2 + was_hit for the at-most-127-step trace.
        key = ((score.to(tl.int64) * 512 + last * 2 + was_hit) * PAGE_STRIDE
               + PAGE_STRIDE - 1 - safe_page) * B + lane
        key = tl.where(occupied, key, lane - B)
        key = tl.where(protected, 1 << 60, key)
        ordered = tl.sort(key, descending=False)
        miss_index = tl.maximum(tl.cumsum(miss.to(tl.int32), axis=0) - 1, 0)
        destination = (tl.gather(ordered, miss_index, axis=0) & (B - 1)).to(tl.int32)
        evicted = tl.gather(slot_page, destination, axis=0)
        tl.store(PageSlot + tl.maximum(evicted, 0), -1, miss & (evicted >= 0))
        tl.store(PageSlot + wanted, destination, miss)
        tl.store(SlotPage + destination, wanted, miss)
        tl.store(Descriptor + 1 + miss_index, wanted, miss)
        tl.store(Dest + miss_index, destination, miss)
        tl.store(Descriptor, tl.sum(miss.to(tl.int32), axis=0))

        if BETA:
            prior_step = tl.load(Last + wanted, query_lane, other=-1)
            prior_bits = tl.load(History + wanted, query_lane, other=0)
            shift = tl.minimum(tl.maximum(step - prior_step, 0), 8)
            tl.store(History + wanted, ((prior_bits << shift) | 1) & 255, query_lane)
        if DELTA:
            # The rank is encoded in the preloaded order of the selected page IDs.
            tl.store(TopRank + wanted, (lane < (K + 3) // 4).to(tl.int32), query_lane)
        tl.store(Last + wanted, step, query_lane)
        tl.store(WasHit + wanted, hit.to(tl.int32), query_lane)

    return resolve


def load_run(bundle, smoke):
    """Read the frozen schedule and ensure it matches the timing contract."""
    with ZipFile(bundle) as archive, tempfile.TemporaryDirectory() as directory:
        manifest = json.loads(archive.read("manifest.json"))
        assert manifest["dtype"] == "torch.bfloat16"
        assert (manifest["page_size"], manifest["top_k"], manifest["max_new_tokens"]) == (16, 64, 128)
        docs = manifest["heldout"]
        assert len(docs) == len({d["source_sha256"] for d in docs}) == 40
        assert len({d["request_id"] for d in docs}) == 40
        strata = Counter((doc["request_id"].split("/")[0], doc["length_band_start"]) for doc in docs)
        assert len(strata) == 4 and set(strata.values()) == {10}
        seen = set()
        requests = []
        path = Path(directory) / "trace.jsonl"
        for i, doc in enumerate(docs, 1):
            request = doc["request_id"]
            stratum = (request.split("/")[0], doc["length_band_start"])
            if smoke and stratum in seen:
                continue
            seen.add(stratum)
            path.write_bytes(archive.read(f"{i:02d}.jsonl"))
            streams = read_trace(path)
            assert set(streams) == {(request, layer) for layer in range(28)}
            assert all(0 < len(steps) < 128 and all(len(p) == TOP_K for p in steps)
                       for steps in streams.values())
            layers = sorted((layer, steps) for (_, layer), steps in streams.items())
            requests.append((request, doc["length_band_start"], layers[:1] if smoke else layers))
    return manifest, requests


class LayerCache:
    def __init__(self, pages):
        import torch

        self.torch = torch
        self.resolve = define_resolver()
        self.pages = pages
        self.stream = torch.cuda.current_stream()
        self.host = torch.empty((pages, *PAGE_SHAPE), dtype=torch.bfloat16, pin_memory=True)
        self.host.uniform_(-1.0, 1.0, generator=torch.Generator().manual_seed(31278))
        self.host_stage = torch.empty((TOP_K, *PAGE_SHAPE), dtype=torch.bfloat16, pin_memory=True)
        self.gpu_stage = torch.empty_like(self.host_stage, device="cuda", pin_memory=False)
        self.slots = torch.empty((BUDGET, *PAGE_SHAPE), device="cuda", dtype=torch.bfloat16)
        self.slot_page = torch.empty(BUDGET, device="cuda", dtype=torch.int32)
        self.page_slot = torch.empty(pages, device="cuda", dtype=torch.int32)
        self.last = torch.empty_like(self.page_slot)
        self.hit = torch.empty_like(self.page_slot)
        self.top = torch.empty_like(self.page_slot)
        self.history = torch.empty_like(self.page_slot)
        self.descriptor = torch.empty(TOP_K + 1, device="cuda", dtype=torch.int64)
        self.host_descriptor = torch.empty(TOP_K + 1, dtype=torch.int64, pin_memory=True)
        self.dest = torch.empty(TOP_K, device="cuda", dtype=torch.int64)
        self.copy_views = [(self.host_descriptor[1:1+n], self.host_stage[:n],
                            self.gpu_stage[:n], self.dest[:n]) for n in range(TOP_K + 1)]
        self.events = [torch.cuda.Event(enable_timing=True) for _ in range(4)]

    def reset(self, beta):
        self.slot_page.fill_(-1)
        self.page_slot.fill_(-1)
        if beta:
            self.last.fill_(-1)
            self.history.zero_()
        self.stream.synchronize()

    def replay(self, queries, policy, *, timing=False, breakdown=False, validate=False):
        torch = self.torch
        delta, beta = POLICIES[policy]
        self.reset(beta)
        records = []
        policy_start, policy_end, copy_start, copy_end = self.events
        # Materialize Python views before timing so input/rank preparation is out of scope.
        for step, wanted in enumerate(queries.unbind()):
            if timing:
                start = perf_counter_ns()
            if breakdown:
                policy_start.record(self.stream)
            self.resolve[(1,)](wanted, self.slot_page, self.page_slot, self.last, self.hit,
                               self.top, self.history, self.descriptor, self.dest, step,
                               self.pages, BUDGET, TOP_K, delta, beta, num_warps=4)
            if breakdown:
                policy_end.record(self.stream)
            self.host_descriptor.copy_(self.descriptor, non_blocking=True)
            self.stream.synchronize()
            misses = int(self.host_descriptor[0])
            if not 0 <= misses <= TOP_K:
                raise RuntimeError(f"Resolver emitted invalid miss count {misses} at step {step}")
            pack_us = None
            if misses:
                page_ids, host_stage, gpu_stage, dest = self.copy_views[misses]
                if breakdown:
                    pack_start = perf_counter_ns()
                torch.index_select(self.host, 0, page_ids, out=host_stage)
                if breakdown:
                    pack_us = (perf_counter_ns() - pack_start) / 1000
                    copy_start.record(self.stream)
                gpu_stage.copy_(host_stage, non_blocking=True)
                if breakdown:
                    copy_end.record(self.stream)
                self.slots.index_copy_(0, dest, gpu_stage)
                self.stream.synchronize()
            wall = (perf_counter_ns() - start) / 1000 if timing else None
            policy_us = policy_start.elapsed_time(policy_end) * 1000 if breakdown else None
            copy_us = (copy_start.elapsed_time(copy_end) * 1000 if misses else 0.0) if breakdown else None
            if breakdown and not misses:
                pack_us = 0.0
            if validate:
                wanted_slots = self.page_slot.index_select(0, wanted.to(torch.int64)).to(torch.int64)
                assert bool((wanted_slots >= 0).all().item())
                assert torch.equal(self.slot_page.index_select(0, wanted_slots), wanted)
                reference = self.host.index_select(0, wanted.cpu().to(torch.int64))
                assert torch.equal(self.slots.index_select(0, wanted_slots).cpu().view(torch.int16),
                                   reference.view(torch.int16))
            records.append((misses, wall, policy_us, pack_us, copy_us))
        return records


def percentiles(values):
    ordered = sorted(values)
    return dict(mean_us=statistics.mean(values), median_us=statistics.median(ordered),
                p95_us=ordered[int(.95 * (len(ordered) - 1))])


def paired_interval(doc_delta, seed):
    groups = defaultdict(list)
    for (_, task, band), (delta_us, steps) in doc_delta.items():
        groups[task, band].append((delta_us, steps))
    rng = random.Random(seed)
    samples = []
    for _ in range(10000):
        picked = [doc for group in groups.values() for doc in rng.choices(group, k=len(group))]
        samples.append(sum(delta for delta, _ in picked) / sum(n for _, n in picked))
    samples.sort()
    return [samples[int(p * (len(samples) - 1))] for p in (.025, .975)]


def run(args):
    import torch
    import triton

    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported(including_emulation=False):
        raise RuntimeError("Select an L4 or A100 Colab GPU before running this benchmark.")
    manifest, requests = load_run(args.bundle, args.smoke)
    max_page = max(max(p) for _, _, layers in requests for _, steps in layers for p in steps)
    cache = LayerCache(max_page + 1)
    args.out.mkdir(parents=True, exist_ok=False)
    with args.bundle.open("rb") as trace_file:
        trace_sha256 = hashlib.file_digest(trace_file, "sha256").hexdigest()
    metadata = dict(gpu=torch.cuda.get_device_name(), torch=torch.__version__, triton=triton.__version__,
                    cuda=torch.version.cuda, mode="smoke" if args.smoke else "full",
                    trace_sha256=trace_sha256,
                    trace_gpu=manifest["gpu"], source_seed=manifest["seed"],
                    repeats=args.repeats, budgets=[BUDGET], policies=POLICIES, page_bytes=PAGE_BYTES,
                    payload="deterministic synthetic BF16 K/V with Qwen3-0.6B layer shape")
    (args.out / "benchmark-manifest.json").write_text(json.dumps(metadata, indent=2) + "\n")
    results = {name: defaultdict(list) for name in POLICIES}
    total_misses = Counter()
    total_compulsory = 0
    doc_delta = defaultdict(lambda: [0.0, 0])
    validated_strata = set()
    validated_divergence = None
    with gzip.open(args.out / "timings.csv.gz", "wt", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(("request", "band", "layer", "phase", "repeat", "policy", "decode_step",
                         "misses", "h2d_bytes", "wall_us", "resolver_cuda_us", "host_pack_us", "h2d_cuda_us"))
        for doc_index, (request, band, layers) in enumerate(requests):
            task = request.split("/")[0]
            for layer_index, (layer, steps) in enumerate(layers):
                queries = torch.tensor(steps, device="cuda", dtype=torch.int32).contiguous()
                references, per_step = {}, {}
                for name, (d,b) in POLICIES.items():
                    counts = []
                    references[name] = replay_stream(
                        steps, BUDGET, "weighted", delta=d, beta=b,
                        on_step=lambda t, pages, missing, victims: counts.append(len(missing)))
                    per_step[name] = counts
                total_compulsory += references["LRU"]["compulsory_misses"]
                total_misses.update({name: value["misses"] for name, value in references.items()})
                diverges = per_step["LRU"] != per_step["rank_delta4_beta2"]
                if (task, band) not in validated_strata or (diverges and validated_divergence is None):
                    # Untimed compilation, warm-up and byte-for-byte direct-index checks.
                    for name in POLICIES:
                        warm = cache.replay(queries, name, validate=True)
                        assert [r[0] for r in warm] == per_step[name]
                    validated_strata.add((task, band))
                    if diverges:
                        validated_divergence = [request, layer]
                names = list(POLICIES)
                for rep in range(args.repeats + 1):
                    breakdown = rep == args.repeats
                    order = names if (doc_index + layer_index + rep) % 2 == 0 else names[::-1]
                    per_policy = {}
                    for name in order:
                        rows = cache.replay(queries, name, timing=not breakdown, breakdown=breakdown)
                        observed = sum(r[0] for r in rows)
                        if [r[0] for r in rows] != per_step[name] or observed != references[name]["misses"]:
                            raise AssertionError(f"{request} layer {layer} {name}: GPU {observed}, CPU {references[name]['misses']}")
                        for step, (misses, wall, resolver, pack, copy) in enumerate(rows):
                            writer.writerow((request, band, layer, "breakdown" if breakdown else "wall",
                                             rep, name, step, misses, misses * PAGE_BYTES,
                                             wall, resolver, pack, copy))
                            if breakdown:
                                results[name]["resolver"].append(resolver)
                                results[name]["host_pack"].append(pack)
                                results[name]["h2d"].append(copy)
                            else:
                                results[name]["wall"].append(wall)
                        if not breakdown:
                            per_policy[name] = sum(r[1] for r in rows)
                    if not breakdown:
                        doc_delta[request, task, band][0] += per_policy[names[1]] - per_policy[names[0]]
                        doc_delta[request, task, band][1] += len(steps)
            f.flush()
            print(f"[{doc_index + 1}/{len(requests)}] {request[:32]} completed", flush=True)

    if not args.smoke:
        assert validated_divergence is not None
        expected = {"LRU": 162369, "rank_delta4_beta2": 161712}
        assert {name: misses - total_compulsory for name, misses in total_misses.items()} == expected
    with (args.out / "paired-documents.csv").open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(("request", "task", "length_band_start", "paired_layer_steps",
                         "rank_minus_lru_mean_wall_us"))
        for (request, task, band), (delta_us, n) in doc_delta.items():
            writer.writerow((request, task, band, n, delta_us / n))
    summary = {}
    for name, values in results.items():
        summary[name] = {key: percentiles(times) for key, times in values.items()}
        summary[name].update(misses=total_misses[name], repeat_misses=total_misses[name]-total_compulsory,
                             h2d_bytes=total_misses[name] * PAGE_BYTES)
    metadata.update(payload_checked_strata=len(validated_strata), payload_checked_divergent_stream=validated_divergence,
                    warmup="untimed payload replay of each smoke stratum, cold reset every policy replay",
                    step_misses_checked_against_cpu=True)
    (args.out / "benchmark-manifest.json").write_text(json.dumps(metadata, indent=2))
    lru, rank = (summary[name] for name in POLICIES)
    summary["rank_minus_lru"] = {
        "wall_mean_us": rank["wall"]["mean_us"] - lru["wall"]["mean_us"],
        "wall_mean_paired_document_ci95_us": paired_interval(doc_delta, manifest["seed"]),
        "resolver_cuda_mean_us": rank["resolver"]["mean_us"] - lru["resolver"]["mean_us"],
        "h2d_cuda_mean_us": rank["h2d"]["mean_us"] - lru["h2d"]["mean_us"],
        "pack_mean_us": rank["host_pack"]["mean_us"] - lru["host_pack"]["mean_us"],
        "diagnostic_pack_h2d_savings_minus_resolver_overhead_us": (
            lru["host_pack"]["mean_us"] + lru["h2d"]["mean_us"]
            - rank["host_pack"]["mean_us"] - rank["h2d"]["mean_us"]
            - (rank["resolver"]["mean_us"] - lru["resolver"]["mean_us"])
        ),
        "misses": rank["misses"] - lru["misses"],
        "h2d_bytes": rank["h2d_bytes"] - lru["h2d_bytes"],
    }
    (args.out / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    print(json.dumps(summary, indent=2))


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("bundle", type=Path)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--repeats", type=int, default=3)
    parser.add_argument("--smoke", action="store_true")
    args = parser.parse_args()
    if args.repeats < 3:
        parser.error("--repeats must be at least 3")
    run(args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile audit.py
"""CPU smoke + independent archive check. Validate a new L4 smoke output with the same keys."""
import csv
import gzip
import io
import json
from collections import Counter
from pathlib import Path
from zipfile import ZipFile

import numpy as np
from gpu_layer_benchmark import BUDGET, PAGE_BYTES, POLICIES, load_run
from replay import replay_stream

KEYS = ("request", "band", "layer", "phase", "repeat", "policy", "decode_step")


def check_timings(rows, expected):
    seen = set()
    for row in rows:
        req, band, layer, phase, rep, policy, step = (row[k] for k in KEYS)
        k = (req, int(band), int(layer), policy, int(step))
        rep = int(rep)
        key = (k, rep)
        assert k in expected and 0 <= rep <= 3 and key not in seen, key
        assert phase == ("wall" if rep < 3 else "breakdown"), key
        assert int(row["misses"]) == expected[k], key
        assert int(row["h2d_bytes"]) == PAGE_BYTES * expected[k], key
        seen.add(key)
    assert len(seen) == 4 * len(expected)
    return len(seen)


def run_smoke_cpu(trace, archive):
    _, docs = load_run(trace, True)
    expected, totals = {}, Counter()
    payload_checks = 0
    for request, band, layers in docs:
        for layer, steps in layers:
            pages = max(max(s) for s in steps) + 1
            # Finite BF16 bit patterns; each comparison covers the complete 64 KiB page.
            rng = np.random.default_rng(31278)
            host = rng.integers(0x3c00, 0x4000, size=(pages, PAGE_BYTES // 2), dtype=np.uint16)
            for policy, (d,b) in POLICIES.items():
                slots = np.empty((BUDGET, PAGE_BYTES // 2), dtype=np.uint16)
                mapping = {}
                free = list(range(BUDGET))
                def inspect(step, ranked, missing, victims):
                    free.extend(mapping.pop(p) for p in victims)
                    for p in missing:
                        mapping[p] = free.pop()
                        slots[mapping[p]] = host[p]
                    assert np.array_equal(slots[[mapping[p] for p in ranked]], host[list(ranked)])
                    expected[request, band, layer, policy, step] = len(missing)
                c = replay_stream(steps, BUDGET, "weighted", delta=d, beta=b, on_step=inspect)
                totals[policy, "misses"] += c["misses"]
                totals[policy, "repeat_misses"] += c["repeat_misses"]
                payload_checks += sum(map(len, steps))
    with ZipFile(archive) as z:
        summary = json.loads(z.read("summary.json"))
        with z.open("timings.csv.gz") as raw, gzip.open(raw, "rt") as f:
            matched = check_timings(csv.DictReader(f), expected)
    for (p,metric), value in totals.items():
        assert summary[p][metric] == value
    return expected, {"documents": len(docs), "payload_pages_checked_cpu": payload_checks,
                      "archive_rows_matched": matched,
                      "counts": {p:{m: totals[p,m] for m in ("misses","repeat_misses")} for p in POLICIES}}

### 1 · 本轮 smoke 验收
先用 CPU 重放四篇，每步逐位检查合成 BF16 页，并与交接包 4,064 条历史 L4 记录核对 miss 和请求的 H2D 字节数。再在当前 L4 上使用 PyTorch BF16 缓冲重新执行 payload 检查和逐步计数。预期两种策略各 488 次 miss、0 次重复 miss。验收通过后才进入完整 A/B。

In [ ]:
from audit import run_smoke_cpu, check_timings
from datetime import datetime, timezone

def run_gpu(mode):
    out = RESULT/mode
    if out.exists():
        raise RuntimeError(f"{out} 已存在。重新执行全部单元格可创建新的独立运行目录。")
    command = [sys.executable, "-u", "gpu_layer_benchmark.py", str(trace),
               "--out", str(out), "--repeats", "3"]
    if mode == "smoke":
        command.append("--smoke")
    with (RESULT/(mode+".log")).open("w") as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, cwd=ROOT) as proc:
            for line in proc.stdout:
                print(line, end="", flush=True)
                log.write(line)
            if proc.wait():
                raise RuntimeError(f"{mode} 未通过。请查看上方日志，或执行最后一格导出诊断包。")
    return json.loads((out/"summary.json").read_text())

expected, smoke_check = run_smoke_cpu(trace, archive)
smoke = run_gpu("smoke")
with gzip.open(RESULT/"smoke/timings.csv.gz","rt") as f:
    smoke_check["new_l4_rows_matched"] = check_timings(csv.DictReader(f), expected)
for p, counts in smoke_check["counts"].items():
    assert all(smoke[p][k] == v for k,v in counts.items())
smoke_check["new_l4_payload_checked"] = True
(RESULT/"smoke-check.json").write_text(json.dumps(smoke_check,indent=2))
print(smoke_check)

### 2 · 完整 A/B
取全部 1,120 个文档 × 层组合。每个组合执行三轮交错的 LRU/rank wall 计时，再执行一轮独立分项计时。每次重置缓存。四个分组首次运行时执行不计时的预热与 payload 检查，并在首个两策略逐步 miss 不同的流上检查 payload。wall 包括 resolver、描述符回传、同步、host pack、H2D 和写入 GPU 槽。分项测量的 resolver 和 H2D 使用 CUDA events，host pack 使用 CPU 时钟。

同一文档的不同层、不同计时轮共享访问轨迹。95% 区间按文档配对、按任务和长度分层，重抽样 10,000 次。用 rank − LRU 的 wall 差及区间判断净收益，分项耗时用于解释原因。

In [ ]:
assert (RESULT/"smoke-check.json").is_file() and smoke_check["new_l4_payload_checked"]
full = run_gpu("full")

### 3 · 查看 L4 实测结果
下表中的 miss 和 H2D 字节按每条流的一次完整回放汇总。重复计时轮只用于估计耗时。

In [ ]:
import pandas as pd
full = json.loads((RESULT/"full/summary.json").read_text())
rows=[]
for policy in ("LRU","rank_delta4_beta2"):
    v=full[policy]
    rows.append(dict(policy=policy,misses=v["misses"],repeat_misses=v["repeat_misses"],
                     H2D_MiB=v["h2d_bytes"]/2**20,
                     **{k+"_us":v[k]["mean_us"] for k in ("resolver","host_pack","h2d","wall")}))
display(pd.DataFrame(rows).set_index("policy").round(3))
delta=full["rank_minus_lru"]
lo,hi=delta["wall_mean_paired_document_ci95_us"]
assert delta["misses"]==-657 and delta["h2d_bytes"]==-657*65536
verdict = ("rank 的单层 cache path 更快" if hi < 0 else
           "rank 的单层 cache path 更慢" if lo > 0 else "区间包含零，尚无法确定净耗时差的方向")
print(f"少搬 {-delta['misses']:,} 页（{-delta['h2d_bytes']/2**20:.4f} MiB）。")
print(f"rank − LRU 同步 wall 差 {delta['wall_mean_us']:+.3f} µs/layer-step，95% CI [{lo:+.3f}, {hi:+.3f}]。{verdict}。")

### 4 · 下载并发回结果
正常完成后，将下载的 **KV_L4_cache_path_results.zip** 发回。包内含 smoke 验收记录、完整逐步计时、按文档配对差、GPU 环境、运行日志和本轮代码。如中途报错，可单独运行这一格导出诊断包。

In [ ]:
for name in ("gpu_layer_benchmark.py","replay.py","audit.py"):
    shutil.copy2(ROOT/name,RESULT/name)
complete = (RESULT/"full/summary.json").is_file() and (RESULT/"smoke-check.json").is_file()
archive_name = "KV_L4_cache_path_results" if complete else "KV_L4_cache_path_diagnostics"
zipped=shutil.make_archive(str(ROOT/archive_name),"zip",RESULT)
print(zipped)
files.download(zipped)